# Assignment 4 - A Cognitive FAQ System Using Pandas (Nova 2.0)
**Jasroop Singh Sachdeva**
Roll Number: 1024170090

In [1]:
import pandas as pd
import string

# show full text in the question/answer columns
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

---
## Q.1 Build your personalized knowledge base

Last two digits of the roll number are used to decide the category of the 2 personalized entries using `["billing", "account", "general"][d % 3]`.

In [2]:
roll_no = "1024170090"

digits = [int(d) for d in roll_no]
last_two = digits[-2:]

category_list = ["billing", "account", "general"]

print("Digits of roll number:", digits)
print("Last two digits:", last_two)
for d in last_two:
    print(f"digit {d} -> category[{d} % 3] = category[{d % 3}] = {category_list[d % 3]}")

Digits of roll number: [1, 0, 2, 4, 1, 7, 0, 0, 9, 0]
Last two digits: [9, 0]
digit 9 -> category[9 % 3] = category[0] = billing
digit 0 -> category[0 % 3] = category[0] = billing


Both digits (9 and 0) give `9 % 3 = 0` and `0 % 3 = 0`, so both personalized entries are **billing** questions.

In [3]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# entries made from last two digits (9 -> billing, 0 -> billing)
personal_entries = [
    {"question": "how do i get a receipt for my payment",
     "answer": "A receipt is sent to your email after every payment. You can also download it from Billing > Payment History.",
     "keywords": "receipt invoice download", "category": category_list[last_two[0] % 3]},
    {"question": "is there a penalty for late payment",
     "answer": "Yes, a late charge of Rs 50 is added if the payment is made after the due date.",
     "keywords": "late penalty due", "category": category_list[last_two[1] % 3]},
]

df = pd.DataFrame(fixed_entries + personal_entries)
print(df)

                                question                                                                                                         answer                  keywords category
0                 what is the annual fee                                                                                      The annual fee is Rs 500.     fee cost price charge  billing
1                  how to reset password                                                                               Go to Settings > Reset Password.      password reset login  account
2            what are your working hours                                                                                      We are open 9 AM to 5 PM.    hours timing open time  general
3                  how can i pay the fee                                                                     You can pay via UPI, card, or net banking.       pay payment upi fee  billing
4  how do i get a receipt for my payment  A receipt is sent to yo

---
## Q.2 Generate and score a hypothesis

Every FAQ entry is treated as a hypothesis for the query. The query is converted to lowercase, punctuation is removed and it is split into words.
- **score** = number of query words that are present in the entry's keywords
- **confidence** = score / total words in the query

Entries with score 0 are dropped and the rest are sorted from highest to lowest confidence. The top one is taken as the answer.

In [4]:
def clean_words(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()


def score_query(query, df):
    query_words = clean_words(query)
    results = []

    for i, row in df.iterrows():
        keywords = row["keywords"].split()
        matched = [w for w in query_words if w in keywords]
        if len(matched) > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched": " ".join(matched),
                "score": len(matched),
                "confidence": round(len(matched) / len(query_words), 2)
            })

    if len(results) == 0:
        return pd.DataFrame()

    ranked = pd.DataFrame(results).sort_values(by="confidence", ascending=False)
    return ranked.reset_index(drop=True)


def answer_query(query, df):
    ranked = score_query(query, df)
    print("Query:", query)
    if ranked.empty:
        print("Sorry, I could not find an answer for that.\n")
        return
    print(ranked[["question", "matched", "score", "confidence"]])
    print("Answer:", ranked.loc[0, "answer"], "\n")

In [5]:
answer_query("How can I pay using UPI?", df)
answer_query("I want to download the invoice of my payment", df)
answer_query("what time do you open", df)
answer_query("do you have a hostel", df)

Query: How can I pay using UPI?
                question  matched  score  confidence
0  how can i pay the fee  pay upi      2        0.33
Answer: You can pay via UPI, card, or net banking. 

Query: I want to download the invoice of my payment
                                question           matched  score  confidence
0  how do i get a receipt for my payment  download invoice      2        0.22
1                  how can i pay the fee           payment      1        0.11
Answer: A receipt is sent to your email after every payment. You can also download it from Billing > Payment History. 

Query: what time do you open
                      question    matched  score  confidence
0  what are your working hours  time open      2         0.4
Answer: We are open 9 AM to 5 PM. 

Query: do you have a hostel
Sorry, I could not find an answer for that.



---
## Q.3 Questions of the same category

Called with the category of the personalized entries (billing).

In [6]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]


personal_category = df.loc[4, "category"]
print("Category:", personal_category)
print(same_category(personal_category, df))

Category: billing
0                   what is the annual fee
3                    how can i pay the fee
4    how do i get a receipt for my payment
5      is there a penalty for late payment
Name: question, dtype: str


---
## Q.4 Add a new keyword and save to CSV

Picked entry 2 (working hours). The keyword is only added if it is not already there.

In [7]:
entry = 2
print("Entry:", df.loc[entry, "question"])
print("Keywords before:", df.loc[entry, "keywords"])

new_keyword = input("Enter a new keyword: ").strip().lower()

if new_keyword in df.loc[entry, "keywords"].split():
    print(new_keyword, "is already a keyword")
else:
    df.loc[entry, "keywords"] = df.loc[entry, "keywords"] + " " + new_keyword

print("Keywords after :", df.loc[entry, "keywords"])

df.to_csv(f"{roll_no}_faq_data.csv", index=False)
print(f"\nSaved to {roll_no}_faq_data.csv")

Entry: what are your working hours
Keywords before: hours timing open time
Enter a new keyword: schedule
Keywords after : hours timing open time schedule

Saved to 1024170090_faq_data.csv


In [8]:
# reading the file back to check it saved properly
print(pd.read_csv(f"{roll_no}_faq_data.csv"))

                                question                                                                                                         answer                         keywords category
0                 what is the annual fee                                                                                      The annual fee is Rs 500.            fee cost price charge  billing
1                  how to reset password                                                                               Go to Settings > Reset Password.             password reset login  account
2            what are your working hours                                                                                      We are open 9 AM to 5 PM.  hours timing open time schedule  general
3                  how can i pay the fee                                                                     You can pay via UPI, card, or net banking.              pay payment upi fee  billing
4  how do i get a receipt for 

---
## Q.5 Number of FAQ entries per category

In [9]:
print(df.groupby("category").size())

category
account    1
billing    4
general    1
dtype: int64


---
## Q.6 Handling ties

`answer_query_v2` uses the same `score_query()` from Q2. If more than one entry has the highest confidence, it prints all of them instead of just picking the first one.

In [10]:
def answer_query_v2(query, df):
    ranked = score_query(query, df)
    print("Query:", query)
    if ranked.empty:
        print("Sorry, I could not find an answer for that.\n")
        return

    best = ranked["confidence"].max()
    top = ranked[ranked["confidence"] == best]

    if len(top) > 1:
        print(f"Tie: {len(top)} entries have the same confidence ({best}). Showing all of them:")
        for i, row in top.iterrows():
            print(f"  - {row['question']} -> {row['answer']}")
    else:
        print("Best match:", top.iloc[0]["question"])
        print("Answer:", top.iloc[0]["answer"])
    print()

In [11]:
# tie - "fee" is a keyword in both fee entries
answer_query_v2("What is the fee?", df)

# no tie - only the password entry matches
answer_query_v2("I forgot my password, how do I reset it", df)

Query: What is the fee?
Tie: 2 entries have the same confidence (0.25). Showing all of them:
  - what is the annual fee -> The annual fee is Rs 500.
  - how can i pay the fee -> You can pay via UPI, card, or net banking.

Query: I forgot my password, how do I reset it
Best match: how to reset password
Answer: Go to Settings > Reset Password.

